# Gene Regulatory Network Inference Pipeline

This notebook orchestrates the entire GRN inference pipeline:
1. **Data Generation**: Generates a scale-free network and harvests time-series data pairs.
2. **Bayesian Inference**: Uses `graph-tool` and our custom `GeneExprBlockState` to run MCMC and infer the posterior ensemble of network topologies and parameters.
3. **Prediction**: Exports the ensemble to raw binaries and calls Julia to run high-performance stationary-state knockout simulations.

In [1]:
import numpy as np
import graph_tool.all as gt
import sys
import os
import subprocess

# Add the parent directory to sys.path so we can import our modules and gene_expr
sys.path.append(os.path.abspath('..'))

from inference_pipeline.data_generation.network_generator import generate_scale_free_W
from inference_pipeline.data_generation.simulator import simulate_realistic_pairs
from inference_pipeline.inference.mcmc_engine import run_burn_in, run_posterior_sampling, compute_ensemble_stats
from inference_pipeline.inference.metrics import plot_probabilistic_network, plot_inferred_vs_true_am
from inference_pipeline.prediction.export_utils import export_ensemble_to_bin, export_true_to_bin
from gene_expr import GeneExprBlockState

## 1. Data Generation

In [3]:
# Configuration
N = 20
m = 1
seed_network = 42

dt_sim = 0.01
dt_sample = 0.5
dt_meas = 0.05
M_pairs = 500
t_window = (0.0, 3.0)

mean_d = 0.5
mean_V = 2.0
D_base = 0.02

# Generate TRUE network
W_true = generate_scale_free_W(N, m, seed=seed_network)
theta_true = [np.log(mean_d), np.log(mean_V), np.log(D_base)]

# Simulate state-transition pairs
s, hub_node, out_degrees, d_true, V_true = simulate_realistic_pairs(
    W_true, M=M_pairs, dt_sim=dt_sim, dt_sample=dt_sample, dt_meas=dt_meas, 
    D_base=D_base, mean_d=mean_d, std_d=0.0, mean_V=mean_V, std_V=0.0, 
    seed=123, t_window=t_window
)

print(f"Harvested {len(s)} pairs. Master Regulator is Node {hub_node}.")

Harvested 500 pairs. Master Regulator is Node 0.


## 2. Bayesian Inference & MCMC

In [4]:
# Instantiate an empty graph in graph-tool format
g_init = gt.Graph(directed=True)
g_init.add_vertex(N)

# Initialize the custom state with an uninformative prior
initial_theta = [0.0, 0.0, np.log(0.05)]
state = GeneExprBlockState(s, dt=dt_meas, g=g_init, state_args=dict(deg_corr=True), theta=initial_theta)
print(f"Initial description length (empty network): {state.entropy():.4f}")

# Phase 1: Burn-in
state = run_burn_in(state, n_burnin=50, optimize_params=True)

# Phase 2: Posterior Sampling
n_samples = 100
W_ensemble, theta_ensemble = run_posterior_sampling(state, N, n_samples=n_samples, delta_k=1, optimize_params=True)

# Compute and save ensemble stats
edge_probabilities, weight_mean, weight_std = compute_ensemble_stats(W_ensemble)

import os
os.makedirs("results", exist_ok=True)
np.savez_compressed(
    "results/grn_ensemble_results.npz", 
    W_ensemble=W_ensemble, 
    theta_ensemble=theta_ensemble,
    edge_probabilities=edge_probabilities,
    weight_mean=weight_mean,
    weight_std=weight_std
)
print("Ensemble results saved to results/grn_ensemble_results.npz!")

Initial description length (empty network): -10748.0361

--- PHASE 1: Equilibration (50 sweeps) ---
Burn-in Sweep  1/50: Entropy = -11734.9845 (dS_topo =   0.0000, dS_param = -986.9483, moves = 15742)
Burn-in Sweep 10/50: Entropy = -18131.9657 (dS_topo = -31.2005, dS_param =  -0.0224, moves = 11203)
Burn-in Sweep 20/50: Entropy = -18383.7034 (dS_topo = -31.8572, dS_param =  -0.0242, moves = 11865)
Burn-in Sweep 30/50: Entropy = -18471.5785 (dS_topo = -10.2447, dS_param =  -0.4068, moves = 15627)
Burn-in Sweep 40/50: Entropy = -18604.2391 (dS_topo = -11.2017, dS_param =  -0.1468, moves = 12386)
Burn-in Sweep 50/50: Entropy = -18634.9913 (dS_topo = -54.6909, dS_param =  -0.6131, moves = 12347)

--- PHASE 2: Posterior Sampling (100 samples, every 1 sweeps) ---
Sample   1/100: Entropy = -18611.6266
Sample  10/100: Entropy = -18614.6749
Sample  20/100: Entropy = -18528.4481
Sample  30/100: Entropy = -18672.0901
Sample  40/100: Entropy = -18556.3036
Sample  50/100: Entropy = -18607.2668
Samp

### Inference Visualizations

In [5]:
plot_probabilistic_network(W_true, edge_probabilities, save_path="images/probabilistic_network.png")
plot_inferred_vs_true_am(W_true, W_ensemble[-1], save_path="images/am_comparison.png")

Saved probabilistic network plot to images/probabilistic_network.png
Saved AM comparison plot to images/am_comparison.png


## 3. Julia Perturbation Analysis

In [7]:
# Export numpy tensors to raw C-order binaries for fast Julia loading
export_ensemble_to_bin("results/grn_ensemble_results.npz", "results/W_ensemble.bin", "results/theta_ensemble.bin")
export_true_to_bin(W_true, theta_true, "results/W_true.bin", "results/theta_true.bin")

# Call the Julia script for knockout prediction and visualization
print(f"Running Julia simulation with Node {hub_node + 1}...")
subprocess.run(["julia", "prediction/predict_knockout.jl", str(hub_node + 1)], check=True)

Exported ensemble arrays to results/W_ensemble.bin and results/theta_ensemble.bin
Exported true arrays to results/W_true.bin and results/theta_true.bin
Running Julia simulation with Node 1...
Detected N = 20 from W_true.bin.
Detected N_samples = 100 from W_ensemble.bin.
Loading ensemble results from binary...
Loading TRUE network...
W_ensemble binary read complete.
Simulating ensemble of 100 perturbed networks...
Simulating ensemble of 100 unperturbed networks...
Simulating TRUE networks (Perturbed and Unperturbed)...
Extracting stationary states...
Plotting Delta Stationary State distributions...
Saved images/delta_stationary_states_plot.png successfully!


CompletedProcess(args=['julia', 'prediction/predict_knockout.jl', '1'], returncode=0)